# Gradient Descent Optimization

Gradient Descent is a foundational optimization algorithm in Machine Learning. While analytical solutions like the Ordinary Least Squares (OLS) closed-form matrix inversion work well for small datasets, they become computationally expensive for high-dimensional data. Gradient Descent provides an iterative approximation technique to find the minimum of a differentiable function, making it highly scalable.

---

## 1. Core Concepts

> **Definition:** Gradient Descent is a first-order iterative optimization algorithm used to find the local minimum of a differentiable loss function by taking steps proportional to the negative of the gradient.

### Key Terms:
- **Optimization:** The process of adjusting model parameters (like weights/slope $m$ and intercept $b$) to minimize the error (loss function).
- **Differentiable Function:** A function for which derivatives can be calculated at any point, providing the slope/gradient to guide the parameters.
- **Generality:** Although demonstrated here using Linear Regression, Gradient Descent is a universal optimizer used in Logistic Regression, Support Vector Machines, Neural Networks, and Deep Learning.

---

## 2. Intuition: The Valley Analogy

Imagine you are blindfolded and placed at the top of a foggy mountain valley. Your goal is to reach the bottom of the valley (the minimum loss).

1. **Assess the Slope:** You feel the ground with your feet to find which direction slopes downward.
2. **Take a Step:** You take a step in the direction of the steepest downward slope (negative gradient).
3. **Adjust Step Size:** The size of your step is controlled by the **Learning Rate** ($\eta$).
4. **Iterate:** You repeat the process until the ground becomes flat, indicating you have reached the bottom.

### The Update Rule
For any parameter $\theta$, the update formula at iteration $t$ is:

$$\theta_{new} = \theta_{old} - \eta \cdot \nabla L(\theta)$$

Where:
- **$\eta$ (Learning Rate):** A hyperparameter determining the size of the step.
- **$\nabla L(\theta)$ (Gradient):** The slope of the loss function at the current parameter value.

### Stopping Criteria:
The iterations stop when:
1. A predefined number of steps (**epochs**) has been reached.
2. The change in the loss function or parameters between consecutive updates becomes extremely small (less than a threshold $\epsilon$).

---

## 3. Mathematical Formulation (Simple Linear Regression)

For simple linear regression, our prediction model is:

$$\hat{y}_i = m x_i + b$$

We use the **Mean Squared Error (MSE)** loss function:

$$L(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - (m x_i + b))^2$$

To find how we must update $m$ and $b$ to reduce the loss, we calculate the partial derivatives of $L$ with respect to both parameters.

### 3.1 Derivative with respect to Intercept ($b$)
Applying the chain rule:

$$\frac{\partial L}{\partial b} = \frac{1}{n} \sum_{i=1}^{n} 2(y_i - (m x_i + b)) \cdot (-1)$$

$$\frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$

### 3.2 Derivative with respect to Slope ($m$)
Applying the chain rule:

$$\frac{\partial L}{\partial m} = \frac{1}{n} \sum_{i=1}^{n} 2(y_i - (m x_i + b)) \cdot (-x_i)$$

$$\frac{\partial L}{\partial m} = -\frac{2}{n} \sum_{i=1}^{n} x_i (y_i - \hat{y}_i)$$

### 3.3 Gradient Update Steps
Using these gradients, we update both $m$ and $b$ simultaneously during each epoch:

$$b_{new} = b_{old} - \eta \cdot \left(-\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)\right)$$

$$m_{new} = m_{old} - \eta \cdot \left(-\frac{2}{n} \sum_{i=1}^{n} x_i (y_i - \hat{y}_i)\right)$$

---

## 4. Code Implementation & Visualization

Let's write a custom Python class to implement Gradient Descent for Simple Linear Regression and visualize how the line and parameters converge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')

# Generate toy dataset
np.random.seed(42)
X = np.random.uniform(1, 5, 50)
y = 2.5 * X + 1.2 + np.random.normal(0, 0.4, 50)

# Visualize dataset
plt.figure(figsize=(7, 4.5))
plt.scatter(X, y, color='#3498db', alpha=0.8, edgecolors='w', label='Data Points')
plt.title('Synthetic Linear Dataset')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()


In [ ]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.m = 0.0  # Initialize slope to 0
        self.b = 0.0  # Initialize intercept to 0
        self.history = []  # To store history for animation/visualization
        
    def fit(self, X, y):
        n = len(X)
        
        for epoch in range(self.epochs):
            # 1. Calculate predictions
            y_pred = self.m * X + self.b
            
            # 2. Compute error / loss (MSE)
            loss = np.mean((y - y_pred)**2)
            self.history.append((self.m, self.b, loss))
            
            # 3. Calculate gradients
            d_b = - (2/n) * np.sum(y - y_pred)
            d_m = - (2/n) * np.sum(X * (y - y_pred))
            
            # 4. Update parameters simultaneously
            self.b = self.b - self.lr * d_b
            self.m = self.m - self.lr * d_m
            
    def predict(self, X):
        return self.m * X + self.b


In [ ]:
# Train our Gradient Descent Regressor
gd_model = GDRegressor(learning_rate=0.05, epochs=150)
gd_model.fit(X, y)

print(f"Trained Slope (m):     {gd_model.m:.4f} (True: 2.5)")
print(f"Trained Intercept (b): {gd_model.b:.4f} (True: 1.2)")


In [ ]:
# Plot model convergence over epochs
epochs_to_plot = [0, 2, 5, 10, 20, 50, 100, 149]
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='#3498db', alpha=0.6, edgecolors='w', label='Data Points')

x_line = np.linspace(1, 5, 100)
for epoch in epochs_to_plot:
    m_epoch, b_epoch, _ = gd_model.history[epoch]
    y_line = m_epoch * x_line + b_epoch
    # Shade of red gets darker as epoch increases
    alpha = 0.3 + 0.7 * (epoch / gd_model.epochs)
    plt.plot(x_line, y_line, color='red', alpha=alpha, label=f'Epoch {epoch}')

plt.title('Best-Fit Line Evolution Across Epochs')
plt.xlabel('X')
plt.ylabel('y')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Visualizing the trajectory of parameter updates on a 2D Contour Plot of the Loss Surface
# Create grid of m and b values
m_vals = np.linspace(-1, 4, 100)
b_vals = np.linspace(-1, 3, 100)
M, B = np.meshgrid(m_vals, b_vals)
Z = np.zeros(M.shape)

# Compute loss for each grid point
for i in range(len(m_vals)):
    for j in range(len(b_vals)):
        pred = M[j, i] * X.reshape(-1, 1) + B[j, i]
        Z[j, i] = np.mean((y.reshape(-1, 1) - pred)**2)

# Plot contours
plt.figure(figsize=(10, 6))
contours = plt.contour(M, B, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=True, fontsize=8)

# Plot trajectory of parameters during training
history = np.array(gd_model.history)
plt.plot(history[:, 0], history[:, 1], 'o-', color='#e74c3c', markersize=3, label='Gradient Descent Path')
plt.scatter([2.5], [1.2], color='black', marker='*', s=150, zorder=5, label='Global Minimum')

plt.title('Loss Surface Contours & Parameter Optimization Path')
plt.xlabel('Slope (m)')
plt.ylabel('Intercept (b)')
plt.legend()
plt.show()


## 5. Critical Factors Influencing Gradient Descent

Several key aspects dictate the success and speed of Gradient Descent optimization.

### 5.1 Learning Rate ($\eta$)
The learning rate dictates the step size at each iteration. It must be selected carefully:
- **Too Small:** Step size is minute, leading to extremely slow convergence. The model requires thousands of epochs to reach the minimum.
- **Too Large:** Step size is massive. The algorithm may overshoot the minimum and bounce back and forth, or even diverge entirely (loss goes to infinity).

| Learning Rate | Speed | Convergence | Risk |
| :--- | :--- | :--- | :--- |
| **Very Low** (e.g., $10^{-5}$) | Slower | Guaranteed (eventually) | High computation time |
| **Optimal** | Fast | Guaranteed | Minimal |
| **Very High** (e.g., $1.0$) | Unstable | Diverges | Divergence / numerical overflow |

In [ ]:
# Demonstrating impact of learning rate
lr_low = GDRegressor(learning_rate=0.001, epochs=100)
lr_optimal = GDRegressor(learning_rate=0.05, epochs=100)
lr_high = GDRegressor(learning_rate=0.25, epochs=100)  # Overshoots / Diverges

lr_low.fit(X, y)
lr_optimal.fit(X, y)
lr_high.fit(X, y)

plt.figure(figsize=(10, 5))
plt.plot([h[2] for h in lr_low.history], label='Low LR (0.001)', color='#f1c40f', linewidth=2)
plt.plot([h[2] for h in lr_optimal.history], label='Optimal LR (0.05)', color='#2ecc71', linewidth=2)
plt.plot([h[2] for h in lr_high.history[:20]], label='High LR (0.25) - First 20 Epochs', color='#e74c3c', linewidth=2)

plt.title('Loss Convergence for Different Learning Rates')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.yscale('log')  # Log scale to handle divergent high loss values
plt.legend()
plt.show()


### 5.2 Loss Function Shape (Convex vs. Non-Convex)
The geometric shape of the loss function determines if Gradient Descent can find the absolute best parameters:

#### 1. Convex Functions (e.g., Mean Squared Error)
- Shaped like a single bowl.
- Has **only one minimum** (the global minimum).
- Gradient descent is guaranteed to find the global minimum regardless of starting parameters (assuming appropriate learning rate).

#### 2. Non-Convex Functions (e.g., Neural Network Loss Surfaces)
- Filled with hills, valleys, and flat regions.
- Contain multiple **local minima** and **saddle points** (flat regions where gradient is zero but not a minimum).
- Gradient descent can easily get stuck in a local minimum, preventing the model from reaching the global optimum.

---

### 5.3 Influence of Data Scaling
Data scaling has a profound impact on the convergence speed of gradient descent:
- **Unscaled Features:** If features have vastly different ranges (e.g., $CGPA \in [0, 10]$ and $Salary \in [0, 100,000]$), the loss surface contour becomes highly **elongated and oval-shaped**.
  - In this scenario, gradient descent will bounce back and forth along the steep axis while making slow progress along the flat axis, requiring thousands of epochs to converge.
- **Scaled Features:** When scaled to similar ranges (e.g., mean centered standard scaling), the loss surface contours become **circular**.
  - The gradient path points directly towards the minimum center, allowing rapid convergence in very few epochs.

> 💡 **Key Rule:** Always apply feature scaling (Standardization/Normalization) before using optimization-based models like Gradient Descent!